# Exercise 1. Prepare Big Data for ML!

## 1.1 Introducing the Data

As mentioned, you will be working with {cite:t}`dugan-etal-2024-raid`'s RAID dataset. The files can be found in `resources/data/raid` on `UCloud` (should be mounted if you followed [Class Setup](class-setup)):
```bash
└── raid
    ├── test.csv
    ├── test_none.csv <-- NEVER EVALUATE ON TEST BEFORE BEING DONE WITH TRAIN!!
    ├── train.csv
    └── train_none.csv
```

You will be working with the `train_none.csv` which contains the following:
```{figure} ../figures/class2/raid_figure.png
---
name: raid-overview
---
Figure modified from {cite:t}`dugan-etal-2024-raid`.
```

`train_none.csv` is a subset of the entire dataset. The full dataset also contains `adversarial attacks` (6.2M examples in total!). For simplicity, we won't be looking at those today.

:::{admonition} What are adversarial attacks?
:class: dropdown, tip
Adversarial attacks are carefully designed modifications to input data (in our case, text) that can cause a classifier to make incorrect predictions. They exploit weaknesses in the model that were not encountered during training. Incorporating such attacks into the training process can improve the classifier’s robustness against unexpected or manipulated inputs in real-world settings.

In {cite:t}`dugan-etal-2024-raid`'s RAID, these include everything from British spelling, article deletions, and mispellings. Read more about it in their paper!

I have downloaded the full dataset `train.csv` for you to explore if you like. It contains 6.2M examples, so consider choosing a larger UCloud machine if it loads slowly.
:::

### Load the Data
Start by importing `pathlib` and `pandas`:

In [117]:
from pathlib import Path
import pandas as pd

Define paths:

In [118]:
# path of notebook
path = Path.cwd()

data_path = path.parents[1] / "resources" / "data" / "raid" / "train_none.csv"

In [119]:
raw_df = pd.read_csv(data_path)

Let's look at how many rows there is in our df:

In [120]:
print(len(raw_df))

467985


### Your Turn: Look at the raw data
```{admonition} HANDS-ON
:class: red

1. Load `raw_df` in your notebook if you haven't already! 
2. Print all column names  `raw_df` 
3. Do you notice any columns that you might not immediately know what corresponds to? Read up on [the column names](https://huggingface.co/datasets/liamdugan/raid#data-fields) before proceeding!
4. From {numref}`raid-overview`, we have gotten an overview of the kinds of LLMs used in this dataset, but what are they called in our dataframe? Find all unique values in the `models` column.
```

#### Print Column Names

```{admonition} HINT
:class: tip, dropdown
Look at the .columns attribute on Pandas - see [docs](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.columns.html) for help
```

Check the solution:

In [121]:
columns_in_df = raw_df.columns.tolist() # you don't technically need .tolist() - it it just to get it in a neat list (try to remove to see effect) 
print(columns_in_df)

['id', 'adv_source_id', 'source_id', 'model', 'decoding', 'repetition_penalty', 'attack', 'domain', 'title', 'prompt', 'generation']


#### Unique Models

```{admonition} HINT
:class: tip, dropdown
What is the LLM column called? You need this, and then you can use the `.unique()` method:
https://pandas.pydata.org/docs/reference/api/pandas.unique.html
```

Solution below:

In [122]:
unique_models = raw_df["model"].unique()
print(unique_models)

['human' 'llama-chat' 'mpt' 'mpt-chat' 'gpt2' 'mistral' 'mistral-chat'
 'gpt3' 'cohere' 'chatgpt' 'gpt4' 'cohere-chat']


### Subset Data
For now, we want to look only at `human` and `chatgpt` generations! Let's use the `isin()` function that we also played with at the end of [Class 1 (Section 2.3) ](23-parts-of-speech-analysis)

In [123]:
df = raw_df[raw_df["model"].isin(["human", "chatgpt"])]

Let's see how many chatgpt and human generations with the `group.by` function, applying `size()` to it:

In [124]:
df.groupby("model").size()

model
chatgpt    26742
human      13371
dtype: int64

:::{admonition} QUESTION
:class: red

Do you know why it is a problem that we have double the amount of `chatgpt` generations?

Consider this with your group and click to reveal answer before proceeding.

```{dropdown} Click to see ANSWER
Classification models generally assume that all classes in a dataset have roughly the same number of examples. Unbalanced classes can cause the classifier to perform poorly on the under-represented class, also called the `minority class` (see {cite:t}`taskiran_comprehensive_2025`).
```
:::

## 1.2 Fixing Unbalanced Classes
As seen on {numref}`raid-overview`, We have more `chatgpt` rows because different generation parameters are used, producing two sets: greedy and sampling.

```{admonition} LLM FRAMING: What is "greedy" and "sampling" ? 
:class: dropdown, fuchsia
In brief, `greedy` and `sampling` are *decoding* methods that determine how an LLM selects words when generating text. You will learn more about them later in the course!
```
For simplicity, we'll just filter away either `greedy` or `sampling` to balance the classes! Ask me if you want another idea to play with!

### Your Turn: Remove `greedy` from df!
:::{admonition} HANDS-ON
:class: red

You have been using `isin()` to "keep" relevant categories (e.g., to keep only `human` and `chatgpt`).
- This time I want you create `df_balanced` by using `isin()` to REMOVE a cateogory (i.e., `NOT in`)

You can look up how to do this here: https://www.geeksforgeeks.org/python/how-to-use-not-in-filter-in-pandas/

:::

Solution below:


In [125]:
# removing greedy (note the tilde ~ means NOT IN, so we want to throw away greedy!)
df_balanced = df[~df["decoding"].isin(["greedy"])].copy() 

# I'm writing copy since I will otherwise get an annoying warning
# See: https://medium.com/@heyamit10/pandas-df-copy-the-ultimate-guide-a9ba18a78cf7

Let's check if it balanced our df:

In [126]:
df_balanced.groupby("model").size()

model
chatgpt    13371
human      13371
dtype: int64

A final thing we'll do is to add a numerical label instead of "model" to make it easier to fit to our classifier. We'll call it `is_human` to make it easily understandable!

In [127]:
df_balanced["is_human"] = df["model"].apply(lambda x: 1 if x == "human" else 0)

## 1.2 Create Training Splits!
When training a ML model, we need to consider three splits of the data:
```{figure} ../figures/class2/train_test.png
---
name: train-test-class2
---
Figure from [Rahul Chavan](https://medium.com/@rahulchavan4894/understanding-train-test-and-validation-dataset-split-in-simple-quick-terms-5a8630fe58c8)
```

Common percentage splits for train, val, and test are 60%, 20%, 20% or 70%, 15%, 15%.

### Install Scikit-Learn
Since the `raid` dataset already has a `test` set, we only need to split our data into `train` and `val`. Firstly, let's install `scikit-learn`: 

In [128]:
%pip install scikit-learn


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Now let's import the `train_test_split`:

In [129]:
from sklearn.model_selection import train_test_split

### Your Turn: Split Data with Scikit-Learn

:::{admonition} HANDS-ON
:class: red
Check the [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) for the `train_test_split`, and then use it to split in your `df` into `X_train, X_val, y_train, y_val`. 

Your `X` variable should be `df["generations"]` and your `y` variable should be `df["is_human"]`. The size of your validation set should be `20%`.

```{dropdown} Why check documentation?
This is how real coders do their work! While ChatGPT might help you with a code snippet, if you want to understand a function OR use it for a more specific case than the general example, the documentation is a great place to be!
```
:::

:::{admonition} Can you help me breakdown the function?
:class: tip, dropdown

Let's look at the docs together:
```{image} ../figures/class2/train_test_split.png
:alt: train_test_split_docs
:width: 500px
```
&nbsp;
- `arrays` refer to the data variables that should be split. 
    - These are typically arrays `X` and `y`, but can also be `pandas` Series object which is what we call a column when selecting it (e.g., `df_balanced["generations"]` for X) or a `pandas`. 
    - X can also be a df with multiple features e.g., `df["average_sentence_length", "mean_dependency_distance"]`. 
- `test_size` and `train_size` refer to the size of the splits. 
    - We typically only set one (as they have to amount to 100%)
- The `random_state` ensures that the randomness can be reproduced (ì.e., `seed`), yielding the same split everytime! 
- `shuffle`: The function defaults to shuffling and in many cases we want this! You therefore don't need to specify it.
- `stratify`: Often you would put the class variable (`y` or whatever you put as `y`) to ensure that the shuffle puts an equal amount of each class in each split!


Finally, the `train_test_split()` **returns** four variables that you typically unpack like:
```python
X_train, X_test, y_train, y_test = train_test_split(...)
```
Note that you need to write `X` as the first argument (`y`) as the second, if you want this order `X_train, X_test, y_train, y_test`
:::

Solution if you are stuck or want to compare:

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
                                                    df["generations"],
                                                    df["is_human"],
                                                    test_size=0.20
                                                    random_state=42
                                                    stratify=df["is_human"]
                                                    )